In [ ]:
!pip install ucimlrepo pandas numpy scipy statsmodels matplotlib seaborn

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

In [ ]:
# Fetch UCI Online Shoppers Purchasing Intention Dataset
dataset = fetch_ucirepo(id=468)

# Features
X = dataset.data.features

# Target
y = dataset.data.targets

# Combine them
df = pd.concat([X, y], axis=1)

# Display first 5 rows
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [ ]:
'''np.random.seed(42)

df["Group"] = np.random.choice(
    ["A", "B"],
    size=len(df)
)'''

'np.random.seed(42)\n\ndf["Group"] = np.random.choice(\n    ["A", "B"],\n    size=len(df)\n)'

In [ ]:
df["Weekend"].value_counts()

,count
Weekend,
False,9462
True,2868


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 12330
Columns: 18


In [ ]:
print("Group A (Weekday):", (df["Weekend"] == False).sum())
print("Group B (Weekend):", (df["Weekend"] == True).sum())

Group A (Weekday): 9462
Group B (Weekend): 2868


In [ ]:
y = df["Revenue"]

In [ ]:
print(df["Revenue"].value_counts())

Revenue
False    10422
True      1908
Name: count, dtype: int64


In [ ]:
df["Conversion"] = df["Revenue"].astype(int)

In [ ]:
df[["Revenue", "Conversion"]].sample(5)

,Revenue,Conversion
10902,False,0
4606,False,0
408,True,1
11429,False,0
8812,False,0


## Question 2 :

- A sucessful conversion is  defined as a shopping session in which Revenue = True, which menas that the visitor made a purchase. A session where Revenue = False, is considered a non-conversion.The Revenue variable is converted to binary COnversion variable where 1 represents a purchase and 0 represents no purchase.

## Question 3:

conversion rate = (No. of conversions) / No. of visitors

In [ ]:
df["Group"] = df["Weekend"].map({
    False: "A",
    True: "B"
})


In [ ]:
df[["Weekend", "Group"]].head(10)

,Weekend,Group
0,False,A
1,False,A
2,False,A
3,False,A
4,True,B
5,False,A
6,False,A
7,True,B
8,False,A
9,False,A


In [ ]:
summary = (
    df.groupby("Group")["Conversion"]
    .agg(
        Visitors="count",
        Conversions="sum"
    )
)

summary["Conversion_Rate"] = (
    summary["Conversions"] / summary["Visitors"]
)

summary["Conversion_Rate_Percent"] = (
    summary["Conversion_Rate"] * 100
)

summary

,Visitors,Conversions,Conversion_Rate,Conversion_Rate_Percent
Group,,,,
A,9462,1409,0.148911,14.891144
B,2868,499,0.173989,17.398884


## Question 4

0.173989 - 0.148911 = 0.025078

 (.025078) * 100 =

 - 2.51 %

 - B had a conversion rate higher than A by 2.51 %

In [ ]:
p_A = summary.loc["A", "Conversion_Rate"]
p_B = summary.loc["B", "Conversion_Rate"]

difference = p_B - p_A

print("Difference:", difference)
print(f"Difference in % points: {difference * 100:.2f}%")

Difference: 0.025077407184341843
Difference in % points: 2.51%


## Question 5

The Conditional Probabilities

In [ ]:
# P(Purchase | Group A)
p_purchase_given_A = df.loc[df["Group"] == "A", "Conversion"].mean()

# P(Purchase | Group B)
p_purchase_given_B = df.loc[df["Group"] == "B", "Conversion"].mean()

print(f"P(Purchase | Group A) = {p_purchase_given_A:.4f}")
print(f"P(Purchase | Group B) = {p_purchase_given_B:.4f}")

print(f"P(Purchase | Group A) = {p_purchase_given_A * 100:.2f}%")
print(f"P(Purchase | Group B) = {p_purchase_given_B * 100:.2f}%")

P(Purchase | Group A) = 0.1489
P(Purchase | Group B) = 0.1740
P(Purchase | Group A) = 14.89%
P(Purchase | Group B) = 17.40%


The above shows that the probability of a visitor making a purchase given that they are in Group A is 14.89% and in Group B is 17.4%.

## Question 6

**Hypotheses**

### Null Hypothesis
There is no difference in the conversion rates between Group A (weekday visitors) and Group B (weekend visitors).

### Alternative Hypothesis
There is a difference in the conversion rates between Group A (weekday visitors) and Group B (weekend visitors).



## Question 7

**Appropriate Hypothesis Test**

The appropriate hypothesis test is a **two-proportion z-test**.

This test compares the proportions of a binary outcome between two groups.

In our analysis:
- Group A = Weekday visitors
- Group B = Weekend visitors
- Outcome = Conversion (1(True) = purchase, 0(False) = no purchase)

## Question 8
**Why the Two-Proportion Z-Test is Appropriate**

The two-proportion z-test is appropriate because we are comparing
the conversion rates of two independent groups, Group A and Group B.

The outcome is binary:
- 1 = the visitor made a purchase
- 0 = the visitor did not make a purchase

Therefore, we are comparing two proportions:
- Group A conversion rate = 14.89%
- Group B conversion rate = 17.40%

The test allows us to determine whether the difference between
these two conversion rates is statistically meaningful.

## Question 9

Condunct the test (z statistic and p value)

In [ ]:
n_A = summary.loc["A", "Visitors"]
x_A = summary.loc["A", "Conversions"]
p_A = summary.loc["A", "Conversion_Rate"]

n_B = summary.loc["B", "Visitors"]
x_B = summary.loc["B", "Conversions"]
p_B = summary.loc["B", "Conversion_Rate"]

print("A:", n_A, x_A, p_A)
print("B:", n_B, x_B, p_B)

A: 9462 1409 0.1489114352145424
B: 2868 499 0.17398884239888424


In [ ]:
p_pool = (x_A + x_B) / (n_A + n_B)

print(f"{p_pool:.5f}")

0.15474


The pooled conversion rate in approximately 15.47%

In [ ]:
# Standard error
se = np.sqrt(
    p_pool * (1 - p_pool) *
    (1/n_A + 1/n_B)
)

print(f"standard error : {se:.5f}")

standard error : 0.00771


In [ ]:
# Z statistic
z_stat = (p_B - p_A) / se

print(f"z statistic : {z_stat:.4f}")

z statistic : 3.2530


## Question 10

In [ ]:
# p value
from scipy.stats import norm

p_value = 2 * (1 - norm.cdf(abs(z_stat)))


print(f"z statistic : {z_stat:.4f}")
print(f"p value : {p_value:.4f}")

z statistic : 3.2530
p value : 0.0011


0.05 is what is normally used and when we compare it to 0.0011, 0.05 is greater than 0.0011 which means that, the conversion rate differs between weekday and weekend visitors.

We observed earlier that the weekend rate is higher , 17.40% > 14.89%, evidence that weekend visitors have a higher conversion rate in this dataset.

- Hence we reject the null hypothesis.

## Question 11

In [ ]:
# for us to get the confidence interval, we use the unpooled standard error

se_ci = np.sqrt(
    (p_A * (1 - p_A) / n_A) +
    (p_B * (1 - p_B) / n_B)
)

print(f"unpooled standard error : {se_ci:.5f}")

unpooled standard error : 0.00797


In [ ]:
z_critical = 0.00797

margin_error = z_critical * se_ci

print("Margin of error:", margin_error)

Margin of error: 6.351281407685012e-05


In [ ]:
z_critical = 1.96

margin_error = z_critical * se_ci

print("Margin of error:", margin_error)

Margin of error: 0.015619211491922991


In [ ]:
difference = p_B - p_A

ci_lower = difference - margin_error
ci_upper = difference + margin_error

print("Lower:", ci_lower)
print("Upper:", ci_upper)

Lower: 0.009458195692418852
Upper: 0.04069661867626483


In [ ]:
print(
    f"95% Confidence Interval = "
    f"{ci_lower * 100:.2f}% to "
    f"{ci_upper * 100:.2f}%"
)

95% Confidence Interval = 0.95% to 4.07%


## Question 12
The estimated difference is 2.51%

With a 95% confidence Interval of approximately =  0.95% to 4.07%

##### Conclusion:
- We estimate that the new website conversion rate is approximately 2.51% higher than the current website conversion rate. A 95% confidence interval suggests that the true difference could reasonably be between 0.95% to 4.07%

## Question 13 

#### Does the evidence support changing from A to B?

"The statistical evidence shows weekend visitors convert significantly higher than weekday visitors. However, this does NOT justify changing to a new website design. Because we split the groups using the Weekend column, we analyzed user behavior, not a website redesign. Weekend shoppers might just have more free time to buy. The data proves a difference in timing, not that a new website caused it."

## Question 14

Statistical Significance vs. Business Significance
"Statistical significance (Is this difference real or just random chance?) :  With our p-value under 0.05, the 2.5% higher weekend conversion is mathematically real.
Business significance answers (Does this difference make us money after paying for the change?):  A result can be statistically real, but if a new website costs roughly $100,000 to build and only brings in $5,000 in extra sales, it’s a bad business decision. Math tells us it's real; business tells us if it's worth it.

# Summary

### Success/conversion outcome.
Revenue outcome.
1. True. Customer made a purchase
2. False. Customer did not make a purchase

### Conversion rate for each group.
No. of conversions/ No. of visitors
Group A = 0.1489 = 14.89%
Group B = 0.1740 = 17.40%
Group B has the higher observed conversion rate. 

### Difference in conversion rates.
17.40 - 14.89
2.51%


### Conditional probabilities.
Group A = 0.1489
Group B = 0.1740
A visitor in Group A has a 14.89% probability of making a purchase, while a visitor in Group B has a 17.40% probability of making a purchase. 

### H₀ and H₁.
### H₀ (null hypothesis).
This means that the probability between the two groups is equal
The conversion rates between Weekday visitors Group A and weekend visitors Group B

### H₁ (alternative hypothesis).
This means there is a difference in the conversion rates between the two groups.

###  Appropriate hypothesis test.
Two-proportion z-test
Explain why your selected test is appropriate for an A/B test.
We are comparing probability outcomes between two groups.
0 = visitor who did not make the purchase. 17.40%
1 = visitor who made a purchase. 14.89%
The test allows us to determine the difference between the two groups.
The outcome is binary.
We want to determine whether the difference between the two proportions is statistically significant 

### Conduct the test.
The statistical analysis was conducted using Python in Visual Studio Code. 

### Report the test statistic and p-value.
Z statistic = 3.2530
P-value = 0.0011
With a significance level of 0.05, our p-value of 0.0011 is less than it. 
This disproves the null hypothesis. 

###  Calculate an appropriate confidence interval for the difference in proportions.
Lower = 0.00945
Upper = 0.041
Significance level = 0.05 

The confidence interval lies between 0.95% and 4.07%
Because the p-value is less than 0.05, 
we reject the null hypothesis. 


### Interpret the confidence interval.
We estimate the new website conversion rate is approximately 2.51 higher than the current website
Our confidence interval is between 0.95% and 4.10%. 
This means the actual improvement from Group B could be between 0.95 and 4.10 percentage points. 
Since the range does not include zero, it supports our conclusion that Group B performs better than Group A.” 


### Decide whether the evidence supports changing from A to B.
The statistical evidence supports the conclusion that Group B has a higher conversion rate than Group A. 
Group A 14.89%
Group B 17.40%
Improvement of 2.51%
P value of 0.0011
95% confidence interval


###  Explain the difference between statistical significance and business significance.
Stats do show significant differences, but business-wise, the dataset is too small, thus limiting its effectiveness to come to an outright decision.
A 2.51 percentage-point improvement could be very valuable for a company with millions of visitors. 
Therefore, statistical significance does not automatically mean that the change is financially worthwhile. 


###  A/B Testing Challenge
Imagine that:
Version B increases the conversion rate by 2 percentage points.
Ask:
Would you automatically recommend Version B?
We would not recommend Version B.

Your answer must consider:
sample size
The sample size is small considering the number of sub-samples.
Uncertainty

###  P-value
We would examine the p-value to determine whether the observed difference provides sufficient statistical evidence against the null hypothesis, which we rejected.
Our p-value of 0.0011 is below 0.05, providing strong statistical evidence of a difference.

### confidence interval
We would examine whether the confidence interval is entirely above zero(2.51%).
This provides evidence that the true improvement is positive.

### practical/business impact
2.51% improvement needs a lot of customers to feel the impact. 
We would not exercise version B due to the small population in the dataset
Final question
What additional information would you want before recommending Version B to the company?
To offer more data on revenue specifically for both versions.
Expected projections on the new version.
Cost metrics of developing the new version.



FINAL PRESENTATION REQUIREMENTS
Report the relevant:
test statistic
Two-proportion z-test: z statistic of 3.25
P-value
0.0011 
It is less than our significance level of 0.05
We reject the null hypothesis. 
degrees of freedom where applicable
confidence interval
0.95% to 4.10%
Explain what the result means in the real-world context.
Group A had a 14.89% conversion rate, while Group B had a 17.40% conversion rate. Therefore, Group B had an observed conversion rate that was 2.51 percentage points higher. 
However, this does not automatically mean the company should change to Version B. The company should also consider whether the 2.51-percentage-point improvement produces enough additional revenue or profit to justify the cost of implementing the new version. 

Give at least one limitation.
The data set seems random rather than a real-life test.
Behavioral change.Does not factor in offers that would maybe attract customers quicker.
The UCI dataset does not actually contain an "original app" and "new app" variable.